Menambahkan kolom audio ke dataframe.

In [1]:
import pandas as pd
import os
import shutil
import fpdf
from fpdf import FPDF
import pdfplumber
import math
root = "../data/sampled"

In [14]:
def csv_to_pdf(file_csv, delim, output_pdf, col_widths, title=None):
    df  = pd.read_csv(file_csv, delimiter=delim).fillna("")
    pdf = FPDF(orientation="L")
    pdf.set_auto_page_break(auto=False)
    pdf.add_page()

    safe  = lambda v: "".join(c if ord(c) < 256 else "?" for c in str(v))
    W     = pdf.w - 2 * pdf.l_margin
    cw    = [w / sum(col_widths) * W for w in col_widths]

    def set_style(bold=False, fill=(255,255,255), text=(0,0,0)):
        pdf.set_font("helvetica", "B" if bold else "", 7)
        pdf.set_fill_color(*fill); pdf.set_text_color(*text)

    def row_h(row):
        cw_ = pdf.get_string_width("A") or 1
        return max(math.ceil(len(safe(v)) / max(1, int(w / cw_))) for v, w in zip(row, cw)) * LH

    def header():
        set_style(bold=True, fill=(52,73,94), text=(255,255,255))
        pdf.set_x(pdf.l_margin)
        [pdf.cell(w, LH, safe(c), border=1, align="C", fill=True) for w, c in zip(cw, df.columns)]
        pdf.ln(); set_style()

    pdf.set_font("helvetica", size=7)
    LH = pdf.font_size * 1.5

    if title:
        set_style(bold=True); pdf.cell(W, 10, safe(title), align="C", ln=True); set_style()

    header()
    for i, (_, row) in enumerate(df.iterrows()):
        rh = row_h(row)
        if pdf.get_y() + rh > pdf.h - pdf.b_margin:
            pdf.add_page(); header()
        y, x = pdf.get_y(), pdf.l_margin
        pdf.set_fill_color(*(245,245,245) if i % 2 == 0 else (255,255,255))
        for v, w in zip(row, cw):
            pdf.set_xy(x, y); pdf.multi_cell(w, LH, safe(v), border=1, fill=True); x += w
        pdf.set_xy(pdf.l_margin, y + rh)

    pdf.output(output_pdf)

In [17]:
csv_to_pdf(
    file_csv="dataMentah.csv",
    delim=";",
    output_pdf="dataMentah.pdf",
    col_widths=[3, 3, 5, 5, 4, 15],  # folder, subfolder, file_name, file_name_stem, id, sentences
)

In [15]:
csv_to_pdf(
    file_csv="dataClean.csv",
    delim=",",
    output_pdf="dataClean.pdf",
    col_widths=[3, 3, 20, 15],  # folder, subfolder, sentences, path
)

In [ ]:
def pdf_to_csv(input_pdf, output_csv):
    all_rows = []

    with pdfplumber.open(input_pdf) as pdf:
        for page in pdf.pages:
            table = page.extract_table()
            if table:
                for row in table:
                    if row:
                        all_rows.append(row)

    if not all_rows:
        print("Tidak ada tabel ditemukan.")
        return

    header = [str(h) if h is not None else f"col_{i}" for i, h in enumerate(all_rows[0])]
    data   = [row for row in all_rows[1:] if row != all_rows[0]]

    df = pd.DataFrame(data, columns=header)
    df = df.applymap(lambda x: str(x).strip() if x is not None else "")
    if "sentences" in df.columns:
        df["sentences"] = df["sentences"].str.replace("\n", " ", regex=True)

    df.to_csv(output_csv, index=False)
    print(f"Selesai: {output_csv} ({len(df)} rows, {len(df.columns)} columns)")

In [22]:
pdf_to_csv('dataClean.pdf','dataCleanfromPDF.csv')

Selesai: dataCleanfromPDF.csv (16025 rows, 4 columns)


In [23]:
pdf_to_csv('dataMentah.pdf','dataMentahfromPDF.csv')

AttributeError: 'DataFrame' object has no attribute 'str'